In [4]:
import torch
import torch.nn as nn 
import tiktoken 

In [8]:
class MultiHeadAttention(nn.Module): 
  def __init__(self, d_in, d_out, context_length, drop_rate, n_heads, qkv_bias=False): 
    super().__init__()
    self.d_in = d_in
    self.d_out = d_out
    self.context_length = context_length
    self.drop_rate = drop_rate
    self.n_heads = n_heads
    self.head_dim = d_out // n_heads 
    assert d_out % n_heads == 0 

    # weight matrices: (8, 16), PyTorch converts to (16, 8) by default
    self.W_Q = nn.Linear(d_in, d_out, bias=qkv_bias)                    
    self.W_K = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_V = nn.Linear(d_in, d_out, bias=qkv_bias)

    # mask fill 
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    # dropout 
    self.dropout = nn.Dropout(drop_rate)

    # output projection 
    self.out_proj = nn.Linear(d_out, d_out)
  
  def forward(self, x):
    batch, num_tokens, d_in = x.shape 

    # reshape qkv matrices
    queries = self.W_Q(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    keys = self.W_K(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)
    values = self.W_V(x).view(batch, num_tokens, self.n_heads, self.head_dim).transpose(2, 1)

    print("queries:", queries.shape)
    print("keys:", keys.shape)
    print("values:", values.shape)
    
    # attention score 
    attn_score = queries @ keys.transpose(-2, -1)
    scaled_score = attn_score / (self.head_dim ** 0.5)

    # applying causal attention with attention mask 
    scaled_score = scaled_score.masked_fill(self.mask[:num_tokens, :num_tokens], float("-inf"))

    attn_weight = torch.softmax(scaled_score, dim=-1)
    attn_weight = self.dropout(attn_weight)

    # return context vector
    context_vec = (attn_weight @ values).transpose(1,2).contiguous().view(batch, num_tokens, self.d_out)

    return self.out_proj(context_vec) 

In [ ]:
torch.manual_seed(420)
x = torch.randn(2, 4, 8)          # b=2, num_tokens=4, d_in=8
attn = MultiHeadAttention(d_in=8, d_out=16, context_length=1024, n_heads=4, drop_rate=0.1)
out = attn(x)
out.shape 


TypeError: MultiHeadAttention.__init__() missing 1 required positional argument: 'n_heads'